# Credit Card Fraud Detection

End-to-end fraud detection pipeline on highly imbalanced transaction data.

**Pipeline:** EDA → preprocessing → stratified split → imbalance handling (SMOTE) → model benchmarking (Logistic Regression, Random Forest, XGBoost, LightGBM) → threshold-aware evaluation (PR-AUC, ROC-AUC, F1, Recall) → feature importance → SHAP explainability.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score, f1_score,
    precision_score, recall_score, ConfusionMatrixDisplay
)

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import shap

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
RANDOM_STATE = 42

## 1. Data Loading

In [ ]:
df = pd.read_csv('creditcard.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum().sum()

## 2. Exploratory Data Analysis

In [ ]:
class_counts = df['Class'].value_counts()
class_pct = df['Class'].value_counts(normalize=True) * 100

print(class_counts)
print(class_pct)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(x='Class', data=df, ax=axes[0])
axes[0].set_title('Class Distribution (Counts)')
axes[0].set_xticklabels(['Legitimate', 'Fraud'])

axes[1].pie(class_counts, labels=['Legitimate', 'Fraud'], autopct='%1.3f%%',
            colors=['#4C72B0', '#C44E52'], startangle=90)
axes[1].set_title('Class Distribution (%)')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['Amount'], bins=50, ax=axes[0], color='#4C72B0')
axes[0].set_title('Transaction Amount Distribution')

sns.histplot(df['Time'], bins=50, ax=axes[1], color='#55A868')
axes[1].set_title('Transaction Time Distribution')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(x='Class', y='Amount', data=df, ax=axes[0])
axes[0].set_title('Amount by Class')
axes[0].set_xticklabels(['Legitimate', 'Fraud'])

sns.boxplot(x='Class', y='Amount', data=df[df['Amount'] < 500], ax=axes[1])
axes[1].set_title('Amount by Class (Amount < 500)')
axes[1].set_xticklabels(['Legitimate', 'Fraud'])
plt.tight_layout()
plt.show()

In [ ]:
corr = df.corr()
plt.figure(figsize=(18, 14))
sns.heatmap(corr, cmap='coolwarm', center=0, linewidths=0.1, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Matrix')
plt.show()

In [ ]:
top_corr = corr['Class'].drop('Class').abs().sort_values(ascending=False).head(10)
plt.figure(figsize=(10, 6))
sns.barplot(x=top_corr.values, y=top_corr.index, palette='viridis')
plt.title('Top 10 Features Correlated with Class')
plt.xlabel('Absolute Correlation')
plt.show()

## 3. Feature Engineering & Preprocessing

In [ ]:
df['Hour'] = (df['Time'] // 3600) % 24

rob_scaler = RobustScaler()
df['Amount_scaled'] = rob_scaler.fit_transform(df['Amount'].values.reshape(-1, 1))
df['Time_scaled'] = rob_scaler.fit_transform(df['Time'].values.reshape(-1, 1))

df.drop(['Time', 'Amount'], axis=1, inplace=True)

feature_cols = [c for c in df.columns if c != 'Class']
X = df[feature_cols]
y = df['Class']

X.head()

## 4. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f'Train shape: {X_train.shape}, Fraud rate: {y_train.mean():.5f}')
print(f'Test shape:  {X_test.shape}, Fraud rate: {y_test.mean():.5f}')

## 5. Handling Class Imbalance (SMOTE)

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print('Before SMOTE:', y_train.value_counts().to_dict())
print('After SMOTE:', pd.Series(y_train_res).value_counts().to_dict())

## 6. Model Training & Benchmarking

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=12, n_jobs=-1, random_state=RANDOM_STATE
    ),
    'XGBoost': XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        eval_metric='aucpr', use_label_encoder=False,
        n_jobs=-1, random_state=RANDOM_STATE
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        n_jobs=-1, random_state=RANDOM_STATE, verbose=-1
    ),
}

results = {}

for name, model in models.items():
    model.fit(X_train_res, y_train_res)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    results[name] = {
        'model': model,
        'y_pred': y_pred,
        'y_proba': y_proba,
        'roc_auc': roc_auc_score(y_test, y_proba),
        'pr_auc': average_precision_score(y_test, y_proba),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
    }

results_df = pd.DataFrame({
    k: {m: v[m] for m in ['roc_auc', 'pr_auc', 'precision', 'recall', 'f1']}
    for k, v in results.items()
}).T.sort_values('pr_auc', ascending=False)

results_df

## 7. Model Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
results_df[['roc_auc', 'pr_auc', 'f1']].plot(kind='bar', ax=ax, colormap='viridis')
ax.set_title('Model Performance Comparison')
ax.set_ylabel('Score')
ax.set_ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={res['roc_auc']:.4f})")

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves')
axes[0].legend()

for name, res in results.items():
    prec, rec, _ = precision_recall_curve(y_test, res['y_proba'])
    axes[1].plot(rec, prec, label=f"{name} (AP={res['pr_auc']:.4f})")

axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves')
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Best Model — Detailed Evaluation

In [ ]:
best_name = results_df.index[0]
best_res = results[best_name]
best_model = best_res['model']

print(f'Best model: {best_name}')
print(classification_report(y_test, best_res['y_pred'], target_names=['Legitimate', 'Fraud']))

In [ ]:
cm = confusion_matrix(y_test, best_res['y_pred'])
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=['Legitimate', 'Fraud']).plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title(f'Confusion Matrix — {best_name}')
plt.show()

## 9. Threshold Tuning

In [ ]:
prec, rec, thresholds = precision_recall_curve(y_test, best_res['y_proba'])
f1_scores = 2 * (prec * rec) / (prec + rec + 1e-9)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f'Optimal threshold: {best_threshold:.4f}')
print(f'Precision: {prec[best_idx]:.4f}, Recall: {rec[best_idx]:.4f}, F1: {f1_scores[best_idx]:.4f}')

plt.figure(figsize=(10, 6))
plt.plot(thresholds, prec[:-1], label='Precision')
plt.plot(thresholds, rec[:-1], label='Recall')
plt.plot(thresholds, f1_scores[:-1], label='F1 Score')
plt.axvline(best_threshold, color='k', linestyle='--', alpha=0.5, label='Optimal Threshold')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Precision / Recall / F1 vs Threshold')
plt.legend()
plt.show()

## 10. Feature Importance

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=feature_cols)
    importances = importances.sort_values(ascending=False).head(15)

    plt.figure(figsize=(10, 8))
    sns.barplot(x=importances.values, y=importances.index, palette='mako')
    plt.title(f'Top 15 Feature Importances — {best_name}')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()

## 11. Model Explainability (SHAP)

In [ ]:
explainer = shap.TreeExplainer(best_model)
sample = X_test.sample(min(2000, len(X_test)), random_state=RANDOM_STATE)
shap_values = explainer.shap_values(sample)

shap.summary_plot(shap_values, sample, plot_type='bar', show=False)
plt.tight_layout()
plt.show()

In [ ]:
shap.summary_plot(shap_values, sample, show=False)
plt.tight_layout()
plt.show()

## 12. Conclusion

- Severe class imbalance (0.172% fraud) requires resampling (SMOTE) and threshold-aware, PR-AUC-driven evaluation rather than plain accuracy.
- Tree-based ensembles (XGBoost / LightGBM / Random Forest) outperform linear baselines on this feature space.
- The optimal decision threshold from the precision-recall curve should replace the default 0.5 cutoff in production, tuned to the business cost of false negatives vs. false positives.
- SHAP values expose which PCA components drive individual fraud predictions, supporting model auditability and regulatory review.

**Next steps:** cost-sensitive learning, ensemble stacking, real-time streaming inference, drift monitoring on incoming transaction distributions.